# 🎬 MovieLens 数据探索与分析

本 notebook 用于探索 MovieLens 数据集，了解数据分布和特征，为推荐系统建模做准备。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# 添加 src 目录
sys.path.append(os.path.join(os.getcwd(), 'src'))

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

from src.data_loader import MovieLensDataLoader

print("导入完成！")

## 1. 数据加载

In [ ]:
# 加载数据
loader = MovieLensDataLoader(data_dir="data")
ratings, movies, tags = loader.load_data()

print(f"评分数据: {len(ratings)} 条")
print(f"电影数据: {len(movies)} 部")
print(f"标签数据: {len(tags)} 条")

## 2. 评分数据分析

In [ ]:
# 查看评分分布
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 评分分布直方图
axes[0].hist(ratings['rating'], bins=10, edgecolor='black', alpha=0.7, color='#2196F3')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('评分分布')
axes[0].grid(axis='y', alpha=0.3)

# 每个用户的评分数量分布
user_counts = ratings.groupby('userId').size()
axes[1].hist(user_counts, bins=50, edgecolor='black', alpha=0.7, color='#4CAF50')
axes[1].set_xlabel('Number of Ratings per User')
axes[1].set_ylabel('Count')
axes[1].set_title('用户评分数量分布')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/data_exploration_1.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. 电影数据分析

In [ ]:
# 电影类型分析
genres_list = []
for genres in movies['genres']:
    if isinstance(genres, str) and genres != '(no genres listed)':
        genres_list.extend(genres.split('|'))

# 统计类型频率
genre_counts = pd.Series(genres_list).value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
genre_counts.plot(kind='bar', ax=ax, color='#FF9800', edgecolor='black', alpha=0.8)
ax.set_xlabel('Genre')
ax.set_ylabel('Count')
ax.set_title('电影类型分布')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('results/data_exploration_2.png', dpi=300, bbox_inches='tight')
plt.show()

print("电影类型统计:")
print(genre_counts)

## 4. 用户-物品矩阵分析

In [ ]:
# 构建用户-物品矩阵
user_item_matrix = loader.build_user_item_matrix(ratings)

print(f"用户-物品矩阵形状: {user_item_matrix.shape}")
print(f"矩阵密度: {1 - user_item_matrix.isna().sum().sum() / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.4f}")

# 可视化矩阵样本（前50x50）
sample_matrix = user_item_matrix.iloc[:50, :50]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(sample_matrix.notna(), cbar=False, ax=ax, cmap='Blues')
ax.set_xlabel('Movies (sample)')
ax.set_ylabel('Users (sample)')
ax.set_title('用户-物品矩阵样本 (蓝色=有评分)')
plt.tight_layout()
plt.savefig('results/data_exploration_3.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. 统计摘要

In [ ]:
print("=" * 50)
print("📊 数据集统计摘要")
print("=" * 50)
print(f"\n用户数量: {user_item_matrix.shape[0]}")
print(f"电影数量: {user_item_matrix.shape[1]}")
print(f"评分总数: {len(ratings)}")
print(f"平均评分: {ratings['rating'].mean():.2f}")
print(f"评分中位数: {ratings['rating'].median()}")
print(f"矩阵密度: {1 - user_item_matrix.isna().sum().sum() / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.6f}")
print(f"\n平均每用户评分数: {len(ratings) / user_item_matrix.shape[0]:.2f}")
print(f"平均每电影被评分数: {len(ratings) / user_item_matrix.shape[1]:.2f}")
print("=" * 50)